In [31]:
import torchvision

In [36]:
'''Train CIFAR10 with PyTorch.'''
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

import os
import argparse

from models import *
from models import MyResnet

# from utils import progress_bar
from torchvision import models


 


In [42]:
# parser = argparse.ArgumentParser(description='PyTorch CIFAR10 Training')
# parser.add_argument('--lr', default=0.1, type=float, help='learning rate')
# parser.add_argument('--resume', '-r', action='store_true',
#                     help='resume from checkpoint')
# args = parser.parse_args()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
print(device)
best_acc = 0  # best test accuracy
start_epoch = 0  # start from epoch 0 or last checkpoint epoch

# Data
print('==> Preparing data..')
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=512, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=500, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

# Model

print('==> Building model..')
# net = VGG('VGG19')
# net = ResNet18()
# net = ResNetCifar10(num_classes= 10)
# net = ResNet34Cifar10(num_classes = 10)
# net = ResNet50Cifar10(num_classes = 10)

# net = PreActResNet18()
# net = GoogLeNet()
# net = DenseNet121()
# net = ResNeXt29_2x64d()
# net = MobileNet()
# net = MobileNetV2()
# net = DPN92()
# net = ShuffleNetG2()
# net = SENet18()
# net = ShuffleNetV2(1)
# net = EfficientNetB0()
# net = RegNetX_200MF()
# net = SimpleDLA()

# for name, para in net.named_parameters():
#     if para.requires_grad and 'linear' not in name:
#         para.requires_grad = False
#     # print("-"*20)
#     # print(f"name: {name}")
#     # print("values: ")
#     # print(para)


net = net.to(device)
if device == 'cuda':
    net = torch.nn.DataParallel(net)
    cudnn.benchmark = True

resume = True
if resume == True:
    # Load checkpoint.
    print('==> Resuming from checkpoint..')
    assert os.path.isdir('checkpoint'), 'Error: no checkpoint directory found!'
    checkpoint = torch.load('./checkpoint/ckpt.pth')
    net.load_state_dict(checkpoint['net'])
    best_acc = checkpoint['acc']
    start_epoch = checkpoint['epoch']

# Hyperparameter
lr = 0.0005
max_epochs = 50
    
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=lr,
                      momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)




cpu
==> Preparing data..
Files already downloaded and verified
Files already downloaded and verified
==> Building model..
==> Resuming from checkpoint..


In [43]:
# Training
def train(epoch):
    print('\nEpoch: %d' % epoch)
    net.train()
    train_loss = 0
    correct = 0
    total = 0
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = net(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

#         progress_bar(batch_idx, len(trainloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d)'
#                      % (train_loss/(batch_idx+1), 100.*correct/total, correct, total))
#         print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss/(batch_idx+1), 100.*correct/total, correct, total))
        print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss/(batch_idx+1), 100.*correct/total, correct, total))

def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = net(inputs)
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

#             progress_bar(batch_idx, len(testloader), 'Loss: %.3f | Acc: %.3f%% (%d/%d)'
#                          % (test_loss/(batch_idx+1), 100.*correct/total, correct, total))
            print('Loss: %.3f | Acc: %.3f%% (%d/%d)' %(test_loss/(batch_idx+1), 100.*correct/total, correct, total))

    # Save checkpoint.
    acc = 100.*correct/total
    if acc > best_acc:
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.isdir('checkpoint'):
            os.mkdir('checkpoint')
        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc
        
        
    return test_loss, correct, total




In [35]:
for epoch in range(start_epoch, start_epoch+300):
#     train_loss, correct, total = train(epoch)
    train(epoch)
    
#     print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss), 100.*correct/total, correct, total)
#     train_loss, correct, total = test(epoch)
    test(epoch)
#     print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss), 100.*correct/total, correct, total)
    
    
    # optimizer.step()
    scheduler.step()
    # optimizer.zero_grad()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch: {epoch}, Learning Rate: {current_lr}")
    #torch.save(net.module.backbone.state_dict(), './Results/resnet18/backbone_pretrained.pth')
    #torch.save(net.module.linear.state_dict(), './Results/resnet18/linear_pretrained.pth') 
    #print("Save ckpt done.")
    
torch.save(net.module.backbone.state_dict(), './Results/resnet18/backbone_pretrained.pth')
torch.save(net.module.linear.state_dict(), './Results/resnet18/linear_pretrained.pth')


Epoch: 0
Loss: 2.467 | Acc: 9.375% (48/512)
Loss: 2.442 | Acc: 9.961% (102/1024)
Loss: 2.440 | Acc: 9.701% (149/1536)
Loss: 2.421 | Acc: 10.156% (208/2048)
Loss: 2.411 | Acc: 10.195% (261/2560)
Loss: 2.407 | Acc: 10.124% (311/3072)
Loss: 2.400 | Acc: 10.017% (359/3584)
Loss: 2.390 | Acc: 10.352% (424/4096)
Loss: 2.380 | Acc: 10.938% (504/4608)
Loss: 2.375 | Acc: 11.016% (564/5120)
Loss: 2.368 | Acc: 11.222% (632/5632)
Loss: 2.364 | Acc: 11.393% (700/6144)
Loss: 2.360 | Acc: 11.463% (763/6656)
Loss: 2.357 | Acc: 11.635% (834/7168)
Loss: 2.354 | Acc: 11.888% (913/7680)
Loss: 2.352 | Acc: 12.085% (990/8192)
Loss: 2.348 | Acc: 12.190% (1061/8704)
Loss: 2.345 | Acc: 12.489% (1151/9216)
Loss: 2.341 | Acc: 12.438% (1210/9728)
Loss: 2.338 | Acc: 12.637% (1294/10240)
Loss: 2.335 | Acc: 12.760% (1372/10752)
Loss: 2.332 | Acc: 12.882% (1451/11264)
Loss: 2.329 | Acc: 13.001% (1531/11776)
Loss: 2.327 | Acc: 13.143% (1615/12288)
Loss: 2.325 | Acc: 13.164% (1685/12800)
Loss: 2.322 | Acc: 13.206% (17

Loss: 1.861 | Acc: 30.364% (13681/45056)
Loss: 1.860 | Acc: 30.401% (13853/45568)
Loss: 1.859 | Acc: 30.484% (14047/46080)
Loss: 1.859 | Acc: 30.510% (14215/46592)
Loss: 1.857 | Acc: 30.571% (14400/47104)
Loss: 1.856 | Acc: 30.597% (14569/47616)
Loss: 1.855 | Acc: 30.658% (14755/48128)
Loss: 1.855 | Acc: 30.678% (14922/48640)
Loss: 1.853 | Acc: 30.745% (15112/49152)
Loss: 1.853 | Acc: 30.767% (15280/49664)
Loss: 1.852 | Acc: 30.784% (15392/50000)
Loss: 1.752 | Acc: 35.600% (178/500)
Loss: 1.738 | Acc: 37.000% (370/1000)
Loss: 1.754 | Acc: 36.333% (545/1500)
Loss: 1.742 | Acc: 36.550% (731/2000)
Loss: 1.736 | Acc: 37.040% (926/2500)
Loss: 1.740 | Acc: 36.767% (1103/3000)
Loss: 1.738 | Acc: 36.886% (1291/3500)
Loss: 1.737 | Acc: 36.825% (1473/4000)
Loss: 1.737 | Acc: 36.733% (1653/4500)
Loss: 1.734 | Acc: 36.780% (1839/5000)
Loss: 1.742 | Acc: 36.673% (2017/5500)
Loss: 1.743 | Acc: 36.450% (2187/6000)
Loss: 1.744 | Acc: 36.538% (2375/6500)
Loss: 1.745 | Acc: 36.200% (2534/7000)
Loss: 1.7

Loss: 1.653 | Acc: 38.672% (10890/28160)
Loss: 1.653 | Acc: 38.693% (11094/28672)
Loss: 1.652 | Acc: 38.682% (11289/29184)
Loss: 1.651 | Acc: 38.712% (11496/29696)
Loss: 1.651 | Acc: 38.741% (11703/30208)
Loss: 1.652 | Acc: 38.669% (11879/30720)
Loss: 1.650 | Acc: 38.694% (12085/31232)
Loss: 1.649 | Acc: 38.684% (12280/31744)
Loss: 1.649 | Acc: 38.684% (12478/32256)
Loss: 1.650 | Acc: 38.599% (12648/32768)
Loss: 1.650 | Acc: 38.633% (12857/33280)
Loss: 1.649 | Acc: 38.627% (13053/33792)
Loss: 1.649 | Acc: 38.619% (13248/34304)
Loss: 1.649 | Acc: 38.646% (13455/34816)
Loss: 1.649 | Acc: 38.638% (13650/35328)
Loss: 1.648 | Acc: 38.655% (13854/35840)
Loss: 1.648 | Acc: 38.628% (14042/36352)
Loss: 1.647 | Acc: 38.647% (14247/36864)
Loss: 1.647 | Acc: 38.648% (14445/37376)
Loss: 1.647 | Acc: 38.669% (14651/37888)
Loss: 1.647 | Acc: 38.667% (14848/38400)
Loss: 1.646 | Acc: 38.662% (15044/38912)
Loss: 1.646 | Acc: 38.672% (15246/39424)
Loss: 1.646 | Acc: 38.664% (15441/39936)
Loss: 1.645 | Ac

Loss: 1.533 | Acc: 43.192% (4644/10752)
Loss: 1.536 | Acc: 43.102% (4855/11264)
Loss: 1.536 | Acc: 43.062% (5071/11776)
Loss: 1.537 | Acc: 43.115% (5298/12288)
Loss: 1.537 | Acc: 43.023% (5507/12800)
Loss: 1.535 | Acc: 43.029% (5728/13312)
Loss: 1.537 | Acc: 42.969% (5940/13824)
Loss: 1.535 | Acc: 43.032% (6169/14336)
Loss: 1.536 | Acc: 43.009% (6386/14848)
Loss: 1.537 | Acc: 42.962% (6599/15360)
Loss: 1.538 | Acc: 42.893% (6808/15872)
Loss: 1.537 | Acc: 42.981% (7042/16384)
Loss: 1.536 | Acc: 42.998% (7265/16896)
Loss: 1.534 | Acc: 43.038% (7492/17408)
Loss: 1.533 | Acc: 43.103% (7724/17920)
Loss: 1.532 | Acc: 43.224% (7967/18432)
Loss: 1.532 | Acc: 43.206% (8185/18944)
Loss: 1.531 | Acc: 43.241% (8413/19456)
Loss: 1.532 | Acc: 43.259% (8638/19968)
Loss: 1.531 | Acc: 43.301% (8868/20480)
Loss: 1.530 | Acc: 43.288% (9087/20992)
Loss: 1.529 | Acc: 43.336% (9319/21504)
Loss: 1.529 | Acc: 43.291% (9531/22016)
Loss: 1.528 | Acc: 43.399% (9777/22528)
Loss: 1.528 | Acc: 43.420% (10004/23040)

Loss: 1.410 | Acc: 48.200% (1687/3500)
Loss: 1.408 | Acc: 47.900% (1916/4000)
Loss: 1.407 | Acc: 48.000% (2160/4500)
Loss: 1.402 | Acc: 48.180% (2409/5000)
Loss: 1.408 | Acc: 48.145% (2648/5500)
Loss: 1.410 | Acc: 47.950% (2877/6000)
Loss: 1.411 | Acc: 48.000% (3120/6500)
Loss: 1.413 | Acc: 47.929% (3355/7000)
Loss: 1.409 | Acc: 48.107% (3608/7500)
Loss: 1.406 | Acc: 48.225% (3858/8000)
Loss: 1.411 | Acc: 48.024% (4082/8500)
Loss: 1.414 | Acc: 48.067% (4326/9000)
Loss: 1.411 | Acc: 48.158% (4575/9500)
Loss: 1.415 | Acc: 47.870% (4787/10000)
Saving..
Epoch: 6, Learning Rate: 0.0009969804777275899

Epoch: 7
Loss: 1.439 | Acc: 44.336% (227/512)
Loss: 1.400 | Acc: 47.168% (483/1024)
Loss: 1.387 | Acc: 48.828% (750/1536)
Loss: 1.406 | Acc: 47.705% (977/2048)
Loss: 1.404 | Acc: 47.461% (1215/2560)
Loss: 1.415 | Acc: 47.266% (1452/3072)
Loss: 1.416 | Acc: 47.433% (1700/3584)
Loss: 1.414 | Acc: 47.485% (1945/4096)
Loss: 1.412 | Acc: 47.439% (2186/4608)
Loss: 1.424 | Acc: 47.168% (2415/5120)
Lo

...

Loss: 0.379 | Acc: 86.640% (41698/48128)
Loss: 0.379 | Acc: 86.639% (42141/48640)
Loss: 0.379 | Acc: 86.631% (42581/49152)
Loss: 0.379 | Acc: 86.624% (43021/49664)
Loss: 0.379 | Acc: 86.608% (43304/50000)
Loss: 0.816 | Acc: 72.800% (364/500)
Loss: 0.818 | Acc: 72.900% (729/1000)
Loss: 0.806 | Acc: 74.400% (1116/1500)
Loss: 0.843 | Acc: 73.750% (1475/2000)
Loss: 0.856 | Acc: 73.520% (1838/2500)
Loss: 0.863 | Acc: 73.367% (2201/3000)
Loss: 0.872 | Acc: 73.229% (2563/3500)
Loss: 0.875 | Acc: 73.000% (2920/4000)
Loss: 0.867 | Acc: 73.533% (3309/4500)
Loss: 0.868 | Acc: 73.300% (3665/5000)
Loss: 0.865 | Acc: 73.200% (4026/5500)
Loss: 0.861 | Acc: 73.433% (4406/6000)
Loss: 0.857 | Acc: 73.631% (4786/6500)
Loss: 0.859 | Acc: 73.486% (5144/7000)
Loss: 0.855 | Acc: 73.627% (5522/7500)
Loss: 0.858 | Acc: 73.625% (5890/8000)
Loss: 0.856 | Acc: 73.624% (6258/8500)
Loss: 0.854 | Acc: 73.667% (6630/9000)
Loss: 0.849 | Acc: 73.758% (7007/9500)
Loss: 0.848 | Acc: 73.820% (7382/10000)
Epoch: 211, Learn

Loss: 0.384 | Acc: 86.351% (26527/30720)
Loss: 0.383 | Acc: 86.395% (26983/31232)
Loss: 0.383 | Acc: 86.360% (27414/31744)
Loss: 0.384 | Acc: 86.353% (27854/32256)
Loss: 0.384 | Acc: 86.325% (28287/32768)
Loss: 0.385 | Acc: 86.283% (28715/33280)
Loss: 0.384 | Acc: 86.299% (29162/33792)
Loss: 0.383 | Acc: 86.340% (29618/34304)
Loss: 0.383 | Acc: 86.322% (30054/34816)
Loss: 0.383 | Acc: 86.348% (30505/35328)
Loss: 0.382 | Acc: 86.345% (30946/35840)
Loss: 0.382 | Acc: 86.345% (31388/36352)
Loss: 0.381 | Acc: 86.366% (31838/36864)
Loss: 0.381 | Acc: 86.390% (32289/37376)
Loss: 0.381 | Acc: 86.373% (32725/37888)
Loss: 0.381 | Acc: 86.385% (33172/38400)
Loss: 0.381 | Acc: 86.390% (33616/38912)
Loss: 0.381 | Acc: 86.399% (34062/39424)
Loss: 0.380 | Acc: 86.433% (34518/39936)
Loss: 0.379 | Acc: 86.454% (34969/40448)
Loss: 0.380 | Acc: 86.462% (35415/40960)
Loss: 0.380 | Acc: 86.463% (35858/41472)
Loss: 0.380 | Acc: 86.431% (36287/41984)
Loss: 0.380 | Acc: 86.429% (36729/42496)
Loss: 0.380 | Ac

Loss: 0.375 | Acc: 86.617% (11087/12800)
Loss: 0.377 | Acc: 86.553% (11522/13312)
Loss: 0.377 | Acc: 86.589% (11970/13824)
Loss: 0.378 | Acc: 86.558% (12409/14336)
Loss: 0.377 | Acc: 86.658% (12867/14848)
Loss: 0.378 | Acc: 86.628% (13306/15360)
Loss: 0.378 | Acc: 86.593% (13744/15872)
Loss: 0.376 | Acc: 86.664% (14199/16384)
Loss: 0.377 | Acc: 86.606% (14633/16896)
Loss: 0.377 | Acc: 86.621% (15079/17408)
Loss: 0.376 | Acc: 86.674% (15532/17920)
Loss: 0.377 | Acc: 86.632% (15968/18432)
Loss: 0.377 | Acc: 86.618% (16409/18944)
Loss: 0.376 | Acc: 86.672% (16863/19456)
Loss: 0.376 | Acc: 86.734% (17319/19968)
Loss: 0.374 | Acc: 86.807% (17778/20480)
Loss: 0.375 | Acc: 86.809% (18223/20992)
Loss: 0.375 | Acc: 86.802% (18666/21504)
Loss: 0.375 | Acc: 86.814% (19113/22016)
Loss: 0.376 | Acc: 86.772% (19548/22528)
Loss: 0.376 | Acc: 86.793% (19997/23040)
Loss: 0.376 | Acc: 86.808% (20445/23552)
Loss: 0.375 | Acc: 86.856% (20901/24064)
Loss: 0.375 | Acc: 86.841% (21342/24576)
Loss: 0.375 | Ac

Loss: 0.871 | Acc: 73.580% (3679/5000)
Loss: 0.868 | Acc: 73.455% (4040/5500)
Loss: 0.864 | Acc: 73.650% (4419/6000)
Loss: 0.860 | Acc: 73.831% (4799/6500)
Loss: 0.863 | Acc: 73.671% (5157/7000)
Loss: 0.859 | Acc: 73.747% (5531/7500)
Loss: 0.862 | Acc: 73.700% (5896/8000)
Loss: 0.860 | Acc: 73.753% (6269/8500)
Loss: 0.858 | Acc: 73.778% (6640/9000)
Loss: 0.853 | Acc: 73.905% (7021/9500)
Loss: 0.852 | Acc: 73.950% (7395/10000)
Epoch: 216, Learning Rate: 1.772129077110091e-05

Epoch: 217
Loss: 0.451 | Acc: 83.594% (428/512)
Loss: 0.401 | Acc: 85.742% (878/1024)
Loss: 0.439 | Acc: 84.701% (1301/1536)
Loss: 0.448 | Acc: 84.277% (1726/2048)
Loss: 0.429 | Acc: 84.766% (2170/2560)
Loss: 0.427 | Acc: 84.896% (2608/3072)
Loss: 0.418 | Acc: 85.268% (3056/3584)
Loss: 0.418 | Acc: 85.376% (3497/4096)
Loss: 0.418 | Acc: 85.547% (3942/4608)


KeyboardInterrupt: 

In [45]:
for epoch in range(start_epoch, start_epoch+130):
#     train_loss, correct, total = train(epoch)
    train(epoch)
    
#     print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss), 100.*correct/total, correct, total)
#     train_loss, correct, total = test(epoch)
    test(epoch)
#     print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss), 100.*correct/total, correct, total)
    
    
    # optimizer.step()
    scheduler.step()
    # optimizer.zero_grad()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch: {epoch}, Learning Rate: {current_lr}")
    #torch.save(net.module.backbone.state_dict(), './Results/resnet18/backbone_pretrained.pth')
    #torch.save(net.module.linear.state_dict(), './Results/resnet18/linear_pretrained.pth')
    torch.save(net.backbone.state_dict(), './Results/resnet50/backbone.pth')
    torch.save(net.linear.state_dict(), './Results/resnet50/linear.pth')
    #print("Save ckpt done.")
    
torch.save(net.module.backbone.state_dict(), './Results/resnet18/backbone_pretrained_fine.pth')
torch.save(net.module.linear.state_dict(), './Results/resnet18/linear_pretrained_fine.pth')


Epoch: 86
Loss: 0.661 | Acc: 78.125% (400/512)
Loss: 0.660 | Acc: 77.344% (792/1024)
Loss: 0.648 | Acc: 77.930% (1197/1536)
Loss: 0.632 | Acc: 78.369% (1605/2048)
Loss: 0.627 | Acc: 78.359% (2006/2560)
Loss: 0.623 | Acc: 78.288% (2405/3072)
Loss: 0.616 | Acc: 78.432% (2811/3584)
Loss: 0.617 | Acc: 78.320% (3208/4096)
Loss: 0.621 | Acc: 78.103% (3599/4608)
Loss: 0.620 | Acc: 78.223% (4005/5120)
Loss: 0.618 | Acc: 78.161% (4402/5632)
Loss: 0.615 | Acc: 78.174% (4803/6144)
Loss: 0.608 | Acc: 78.305% (5212/6656)
Loss: 0.608 | Acc: 78.348% (5616/7168)
Loss: 0.606 | Acc: 78.424% (6023/7680)
Loss: 0.606 | Acc: 78.503% (6431/8192)
Loss: 0.609 | Acc: 78.424% (6826/8704)
Loss: 0.610 | Acc: 78.353% (7221/9216)
Loss: 0.611 | Acc: 78.351% (7622/9728)
Loss: 0.608 | Acc: 78.564% (8045/10240)
Loss: 0.606 | Acc: 78.637% (8455/10752)
Loss: 0.609 | Acc: 78.516% (8844/11264)
Loss: 0.608 | Acc: 78.550% (9250/11776)
Loss: 0.608 | Acc: 78.630% (9662/12288)
Loss: 0.609 | Acc: 78.641% (10066/12800)
Loss: 0.60

Loss: 0.616 | Acc: 78.077% (33979/43520)
Loss: 0.616 | Acc: 78.055% (34369/44032)
Loss: 0.615 | Acc: 78.069% (34775/44544)
Loss: 0.615 | Acc: 78.098% (35188/45056)
Loss: 0.615 | Acc: 78.103% (35590/45568)
Loss: 0.615 | Acc: 78.069% (35974/46080)
Loss: 0.614 | Acc: 78.097% (36387/46592)
Loss: 0.614 | Acc: 78.131% (36803/47104)
Loss: 0.614 | Acc: 78.104% (37190/47616)
Loss: 0.614 | Acc: 78.108% (37592/48128)
Loss: 0.615 | Acc: 78.080% (37978/48640)
Loss: 0.616 | Acc: 78.064% (38370/49152)
Loss: 0.615 | Acc: 78.061% (38768/49664)
Loss: 0.616 | Acc: 78.048% (39024/50000)
Loss: 0.814 | Acc: 71.800% (359/500)
Loss: 0.777 | Acc: 73.000% (730/1000)
Loss: 0.776 | Acc: 72.533% (1088/1500)
Loss: 0.805 | Acc: 71.750% (1435/2000)
Loss: 0.811 | Acc: 72.000% (1800/2500)
Loss: 0.811 | Acc: 71.800% (2154/3000)
Loss: 0.815 | Acc: 72.000% (2520/3500)
Loss: 0.812 | Acc: 72.225% (2889/4000)
Loss: 0.803 | Acc: 72.800% (3276/4500)
Loss: 0.800 | Acc: 72.940% (3647/5000)
Loss: 0.798 | Acc: 72.764% (4002/5500)


Loss: 0.609 | Acc: 78.477% (20492/26112)
Loss: 0.608 | Acc: 78.478% (20894/26624)
Loss: 0.607 | Acc: 78.490% (21299/27136)
Loss: 0.607 | Acc: 78.451% (21690/27648)
Loss: 0.606 | Acc: 78.477% (22099/28160)
Loss: 0.605 | Acc: 78.526% (22515/28672)
Loss: 0.606 | Acc: 78.516% (22914/29184)
Loss: 0.607 | Acc: 78.479% (23305/29696)
Loss: 0.607 | Acc: 78.483% (23708/30208)
Loss: 0.607 | Acc: 78.496% (24114/30720)
Loss: 0.607 | Acc: 78.493% (24515/31232)
Loss: 0.607 | Acc: 78.503% (24920/31744)
Loss: 0.607 | Acc: 78.463% (25309/32256)
Loss: 0.607 | Acc: 78.506% (25725/32768)
Loss: 0.607 | Acc: 78.504% (26126/33280)
Loss: 0.608 | Acc: 78.504% (26528/33792)
Loss: 0.609 | Acc: 78.440% (26908/34304)
Loss: 0.610 | Acc: 78.392% (27293/34816)
Loss: 0.610 | Acc: 78.346% (27678/35328)
Loss: 0.611 | Acc: 78.312% (28067/35840)
Loss: 0.610 | Acc: 78.312% (28468/36352)
Loss: 0.609 | Acc: 78.339% (28879/36864)
Loss: 0.608 | Acc: 78.376% (29294/37376)
Loss: 0.608 | Acc: 78.399% (29704/37888)
Loss: 0.608 | Ac

Loss: 0.596 | Acc: 79.004% (6472/8192)
Loss: 0.595 | Acc: 79.044% (6880/8704)
Loss: 0.595 | Acc: 79.069% (7287/9216)
Loss: 0.595 | Acc: 79.143% (7699/9728)
Loss: 0.595 | Acc: 79.131% (8103/10240)
Loss: 0.596 | Acc: 79.157% (8511/10752)
Loss: 0.596 | Acc: 79.093% (8909/11264)
Loss: 0.597 | Acc: 79.093% (9314/11776)
Loss: 0.599 | Acc: 79.012% (9709/12288)
Loss: 0.600 | Acc: 78.992% (10111/12800)
Loss: 0.599 | Acc: 79.019% (10519/13312)
Loss: 0.601 | Acc: 78.906% (10908/13824)
Loss: 0.599 | Acc: 78.962% (11320/14336)
Loss: 0.599 | Acc: 79.001% (11730/14848)
Loss: 0.602 | Acc: 78.887% (12117/15360)
Loss: 0.602 | Acc: 78.881% (12520/15872)
Loss: 0.599 | Acc: 78.973% (12939/16384)
Loss: 0.597 | Acc: 79.090% (13363/16896)
Loss: 0.597 | Acc: 79.096% (13769/17408)
Loss: 0.596 | Acc: 79.051% (14166/17920)
Loss: 0.598 | Acc: 78.982% (14558/18432)
Loss: 0.597 | Acc: 78.975% (14961/18944)
Loss: 0.598 | Acc: 78.922% (15355/19456)
Loss: 0.596 | Acc: 78.986% (15772/19968)
Loss: 0.594 | Acc: 79.053% (1

Loss: 0.806 | Acc: 72.000% (360/500)
Loss: 0.780 | Acc: 73.800% (738/1000)
Loss: 0.772 | Acc: 73.800% (1107/1500)
Loss: 0.797 | Acc: 73.200% (1464/2000)
Loss: 0.803 | Acc: 72.960% (1824/2500)
Loss: 0.802 | Acc: 72.833% (2185/3000)
Loss: 0.803 | Acc: 72.914% (2552/3500)
Loss: 0.801 | Acc: 72.975% (2919/4000)
Loss: 0.793 | Acc: 73.444% (3305/4500)
Loss: 0.793 | Acc: 73.460% (3673/5000)
Loss: 0.791 | Acc: 73.327% (4033/5500)
Loss: 0.791 | Acc: 73.417% (4405/6000)
Loss: 0.793 | Acc: 73.231% (4760/6500)
Loss: 0.799 | Acc: 73.071% (5115/7000)
Loss: 0.798 | Acc: 73.040% (5478/7500)
Loss: 0.800 | Acc: 73.100% (5848/8000)
Loss: 0.801 | Acc: 72.976% (6203/8500)
Loss: 0.802 | Acc: 72.967% (6567/9000)
Loss: 0.800 | Acc: 73.063% (6941/9500)
Loss: 0.799 | Acc: 73.110% (7311/10000)
Saving..
Epoch: 92, Learning Rate: 0.00047620676311650495

Epoch: 93
Loss: 0.514 | Acc: 81.641% (418/512)
Loss: 0.564 | Acc: 80.664% (826/1024)
Loss: 0.575 | Acc: 80.469% (1236/1536)
Loss: 0.579 | Acc: 79.932% (1637/2048)


...

Loss: 0.384 | Acc: 86.167% (32647/37888)
Loss: 0.383 | Acc: 86.174% (33091/38400)
Loss: 0.383 | Acc: 86.197% (33541/38912)
Loss: 0.383 | Acc: 86.201% (33984/39424)
Loss: 0.384 | Acc: 86.173% (34414/39936)
Loss: 0.384 | Acc: 86.148% (34845/40448)
Loss: 0.384 | Acc: 86.172% (35296/40960)
Loss: 0.383 | Acc: 86.191% (35745/41472)
Loss: 0.383 | Acc: 86.188% (36185/41984)
Loss: 0.383 | Acc: 86.189% (36627/42496)
Loss: 0.384 | Acc: 86.186% (37067/43008)
Loss: 0.383 | Acc: 86.202% (37515/43520)
Loss: 0.384 | Acc: 86.174% (37944/44032)
Loss: 0.383 | Acc: 86.175% (38386/44544)
Loss: 0.384 | Acc: 86.151% (38816/45056)
Loss: 0.384 | Acc: 86.137% (39251/45568)
Loss: 0.384 | Acc: 86.146% (39696/46080)
Loss: 0.384 | Acc: 86.186% (40156/46592)
Loss: 0.384 | Acc: 86.192% (40600/47104)
Loss: 0.384 | Acc: 86.192% (41041/47616)
Loss: 0.385 | Acc: 86.179% (41476/48128)
Loss: 0.384 | Acc: 86.184% (41920/48640)
Loss: 0.384 | Acc: 86.184% (42361/49152)
Loss: 0.383 | Acc: 86.195% (42808/49664)
Loss: 0.384 | Ac

Loss: 0.368 | Acc: 86.997% (17817/20480)
Loss: 0.368 | Acc: 86.986% (18260/20992)
Loss: 0.368 | Acc: 86.970% (18702/21504)
Loss: 0.369 | Acc: 86.978% (19149/22016)
Loss: 0.369 | Acc: 86.936% (19585/22528)
Loss: 0.368 | Acc: 86.984% (20041/23040)
Loss: 0.367 | Acc: 87.046% (20501/23552)
Loss: 0.366 | Acc: 87.068% (20952/24064)
Loss: 0.367 | Acc: 87.065% (21397/24576)
Loss: 0.367 | Acc: 87.014% (21830/25088)
Loss: 0.367 | Acc: 87.070% (22290/25600)
Loss: 0.366 | Acc: 87.083% (22739/26112)
Loss: 0.367 | Acc: 87.061% (23179/26624)
Loss: 0.368 | Acc: 87.025% (23615/27136)
Loss: 0.368 | Acc: 86.965% (24044/27648)
Loss: 0.370 | Acc: 86.900% (24471/28160)
Loss: 0.371 | Acc: 86.855% (24903/28672)
Loss: 0.371 | Acc: 86.835% (25342/29184)
Loss: 0.370 | Acc: 86.853% (25792/29696)
Loss: 0.370 | Acc: 86.854% (26237/30208)
Loss: 0.370 | Acc: 86.862% (26684/30720)
Loss: 0.371 | Acc: 86.837% (27121/31232)
Loss: 0.370 | Acc: 86.829% (27563/31744)
Loss: 0.371 | Acc: 86.815% (28003/32256)
Loss: 0.372 | Ac

Loss: 0.368 | Acc: 86.367% (2211/2560)
Loss: 0.365 | Acc: 86.556% (2659/3072)
Loss: 0.362 | Acc: 86.802% (3111/3584)
Loss: 0.360 | Acc: 87.036% (3565/4096)
Loss: 0.353 | Acc: 87.370% (4026/4608)
Loss: 0.357 | Acc: 87.207% (4465/5120)
Loss: 0.359 | Acc: 87.234% (4913/5632)
Loss: 0.360 | Acc: 87.191% (5357/6144)
Loss: 0.364 | Acc: 87.064% (5795/6656)
Loss: 0.365 | Acc: 86.956% (6233/7168)
Loss: 0.364 | Acc: 86.992% (6681/7680)
Loss: 0.364 | Acc: 87.012% (7128/8192)
Loss: 0.363 | Acc: 87.144% (7585/8704)
Loss: 0.363 | Acc: 87.142% (8031/9216)
Loss: 0.364 | Acc: 87.161% (8479/9728)
Loss: 0.364 | Acc: 87.148% (8924/10240)
Loss: 0.361 | Acc: 87.221% (9378/10752)
Loss: 0.360 | Acc: 87.225% (9825/11264)
Loss: 0.360 | Acc: 87.313% (10282/11776)
Loss: 0.360 | Acc: 87.305% (10728/12288)
Loss: 0.362 | Acc: 87.219% (11164/12800)
Loss: 0.360 | Acc: 87.290% (11620/13312)
Loss: 0.361 | Acc: 87.290% (12067/13824)
Loss: 0.361 | Acc: 87.263% (12510/14336)
Loss: 0.360 | Acc: 87.311% (12964/14848)
Loss: 0.

Loss: 0.366 | Acc: 87.002% (39645/45568)
Loss: 0.367 | Acc: 86.975% (40078/46080)
Loss: 0.367 | Acc: 86.996% (40533/46592)
Loss: 0.366 | Acc: 87.010% (40985/47104)
Loss: 0.367 | Acc: 87.000% (41426/47616)
Loss: 0.367 | Acc: 86.993% (41868/48128)
Loss: 0.367 | Acc: 86.978% (42306/48640)
Loss: 0.366 | Acc: 86.983% (42754/49152)
Loss: 0.366 | Acc: 86.991% (43203/49664)
Loss: 0.366 | Acc: 86.986% (43493/50000)
Loss: 0.849 | Acc: 73.000% (365/500)
Loss: 0.824 | Acc: 74.600% (746/1000)
Loss: 0.818 | Acc: 75.200% (1128/1500)
Loss: 0.856 | Acc: 74.100% (1482/2000)
Loss: 0.882 | Acc: 74.200% (1855/2500)
Loss: 0.874 | Acc: 74.500% (2235/3000)
Loss: 0.880 | Acc: 74.429% (2605/3500)
Loss: 0.872 | Acc: 74.725% (2989/4000)
Loss: 0.861 | Acc: 74.956% (3373/4500)
Loss: 0.859 | Acc: 74.860% (3743/5000)
Loss: 0.857 | Acc: 74.691% (4108/5500)
Loss: 0.856 | Acc: 74.700% (4482/6000)
Loss: 0.856 | Acc: 74.646% (4852/6500)
Loss: 0.862 | Acc: 74.457% (5212/7000)
Loss: 0.861 | Acc: 74.560% (5592/7500)
Loss: 0.

AttributeError: 'ResNet34Cifar10' object has no attribute 'module'

In [39]:
for epoch in range(start_epoch, start_epoch+300):
#     train_loss, correct, total = train(epoch)
    train(epoch)
    
#     print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss), 100.*correct/total, correct, total)
#     train_loss, correct, total = test(epoch)
    test(epoch)
#     print('Loss: %.3f | Acc: %.3f%% (%d/%d)'% (train_loss), 100.*correct/total, correct, total)
    
    
    # optimizer.step()
    scheduler.step()
    # optimizer.zero_grad()
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch: {epoch}, Learning Rate: {current_lr}")
    #torch.save(net.module.backbone.state_dict(), './Results/resnet18/backbone_pretrained.pth')
    #torch.save(net.module.linear.state_dict(), './Results/resnet18/linear_pretrained.pth')
    torch.save(net.backbone.state_dict(), './Results/resnet18/backbone_pretrained_fine.pth')
    torch.save(net.linear.state_dict(), './Results/resnet18/linear_pretrained_fine.pth')
    #print("Save ckpt done.")
    
torch.save(net.module.backbone.state_dict(), './Results/resnet18/backbone_pretrained_fine.pth')
torch.save(net.module.linear.state_dict(), './Results/resnet18/linear_pretrained_fine.pth')


Epoch: 0
Loss: 2.461 | Acc: 8.789% (45/512)
Loss: 2.453 | Acc: 9.277% (95/1024)
Loss: 2.449 | Acc: 9.375% (144/1536)
Loss: 2.434 | Acc: 9.326% (191/2048)
Loss: 2.425 | Acc: 9.688% (248/2560)
Loss: 2.408 | Acc: 10.124% (311/3072)
Loss: 2.400 | Acc: 10.045% (360/3584)
Loss: 2.392 | Acc: 10.376% (425/4096)
Loss: 2.387 | Acc: 10.286% (474/4608)
Loss: 2.383 | Acc: 10.352% (530/5120)
Loss: 2.380 | Acc: 10.281% (579/5632)
Loss: 2.377 | Acc: 10.531% (647/6144)
Loss: 2.371 | Acc: 10.817% (720/6656)
Loss: 2.364 | Acc: 10.965% (786/7168)
Loss: 2.361 | Acc: 11.146% (856/7680)
Loss: 2.357 | Acc: 11.292% (925/8192)
Loss: 2.353 | Acc: 11.512% (1002/8704)
Loss: 2.349 | Acc: 11.675% (1076/9216)
Loss: 2.345 | Acc: 11.852% (1153/9728)
Loss: 2.342 | Acc: 12.061% (1235/10240)
Loss: 2.339 | Acc: 12.128% (1304/10752)
Loss: 2.337 | Acc: 12.331% (1389/11264)
Loss: 2.333 | Acc: 12.576% (1481/11776)
Loss: 2.331 | Acc: 12.712% (1562/12288)
Loss: 2.329 | Acc: 12.836% (1643/12800)
Loss: 2.326 | Acc: 12.943% (1723/

Loss: 1.882 | Acc: 29.912% (13477/45056)
Loss: 1.882 | Acc: 29.984% (13663/45568)
Loss: 1.880 | Acc: 30.037% (13841/46080)
Loss: 1.879 | Acc: 30.072% (14011/46592)
Loss: 1.878 | Acc: 30.125% (14190/47104)
Loss: 1.877 | Acc: 30.150% (14356/47616)
Loss: 1.876 | Acc: 30.140% (14506/48128)
Loss: 1.875 | Acc: 30.169% (14674/48640)
Loss: 1.874 | Acc: 30.180% (14834/49152)
Loss: 1.873 | Acc: 30.235% (15016/49664)
Loss: 1.873 | Acc: 30.254% (15127/50000)
Loss: 1.722 | Acc: 36.800% (184/500)
Loss: 1.717 | Acc: 37.000% (370/1000)
Loss: 1.732 | Acc: 36.800% (552/1500)
Loss: 1.720 | Acc: 37.350% (747/2000)
Loss: 1.715 | Acc: 37.240% (931/2500)
Loss: 1.728 | Acc: 36.700% (1101/3000)
Loss: 1.731 | Acc: 36.457% (1276/3500)
Loss: 1.727 | Acc: 36.225% (1449/4000)
Loss: 1.723 | Acc: 36.400% (1638/4500)
Loss: 1.718 | Acc: 36.560% (1828/5000)
Loss: 1.723 | Acc: 36.345% (1999/5500)
Loss: 1.721 | Acc: 36.350% (2181/6000)
Loss: 1.722 | Acc: 36.262% (2357/6500)
Loss: 1.723 | Acc: 36.243% (2537/7000)
Loss: 1.7

Loss: 1.630 | Acc: 39.712% (11183/28160)
Loss: 1.629 | Acc: 39.771% (11403/28672)
Loss: 1.630 | Acc: 39.744% (11599/29184)
Loss: 1.630 | Acc: 39.719% (11795/29696)
Loss: 1.629 | Acc: 39.778% (12016/30208)
Loss: 1.628 | Acc: 39.824% (12234/30720)
Loss: 1.627 | Acc: 39.821% (12437/31232)
Loss: 1.627 | Acc: 39.803% (12635/31744)
Loss: 1.628 | Acc: 39.810% (12841/32256)
Loss: 1.628 | Acc: 39.816% (13047/32768)
Loss: 1.628 | Acc: 39.838% (13258/33280)
Loss: 1.627 | Acc: 39.867% (13472/33792)
Loss: 1.627 | Acc: 39.873% (13678/34304)
Loss: 1.625 | Acc: 39.944% (13907/34816)
Loss: 1.624 | Acc: 39.929% (14106/35328)
Loss: 1.624 | Acc: 39.953% (14319/35840)
Loss: 1.625 | Acc: 39.962% (14527/36352)
Loss: 1.624 | Acc: 39.977% (14737/36864)
Loss: 1.624 | Acc: 39.980% (14943/37376)
Loss: 1.623 | Acc: 40.005% (15157/37888)
Loss: 1.624 | Acc: 39.995% (15358/38400)
Loss: 1.623 | Acc: 39.988% (15560/38912)
Loss: 1.622 | Acc: 40.026% (15780/39424)
Loss: 1.623 | Acc: 40.049% (15994/39936)
Loss: 1.623 | Ac

Loss: 1.509 | Acc: 44.234% (4756/10752)
Loss: 1.510 | Acc: 44.203% (4979/11264)
Loss: 1.512 | Acc: 44.090% (5192/11776)
Loss: 1.509 | Acc: 44.206% (5432/12288)
Loss: 1.507 | Acc: 44.266% (5666/12800)
Loss: 1.506 | Acc: 44.336% (5902/13312)
Loss: 1.506 | Acc: 44.459% (6146/13824)
Loss: 1.506 | Acc: 44.385% (6363/14336)
Loss: 1.506 | Acc: 44.397% (6592/14848)
Loss: 1.506 | Acc: 44.342% (6811/15360)
Loss: 1.506 | Acc: 44.342% (7038/15872)
Loss: 1.505 | Acc: 44.354% (7267/16384)
Loss: 1.505 | Acc: 44.389% (7500/16896)
Loss: 1.502 | Acc: 44.503% (7747/17408)
Loss: 1.504 | Acc: 44.448% (7965/17920)
Loss: 1.506 | Acc: 44.450% (8193/18432)
Loss: 1.504 | Acc: 44.552% (8440/18944)
Loss: 1.506 | Acc: 44.572% (8672/19456)
Loss: 1.507 | Acc: 44.486% (8883/19968)
Loss: 1.507 | Acc: 44.487% (9111/20480)
Loss: 1.507 | Acc: 44.474% (9336/20992)
Loss: 1.507 | Acc: 44.522% (9574/21504)
Loss: 1.507 | Acc: 44.572% (9813/22016)
Loss: 1.506 | Acc: 44.598% (10047/22528)
Loss: 1.506 | Acc: 44.653% (10288/23040

Loss: 1.391 | Acc: 48.771% (1707/3500)
Loss: 1.387 | Acc: 48.900% (1956/4000)
Loss: 1.385 | Acc: 49.178% (2213/4500)
Loss: 1.382 | Acc: 49.300% (2465/5000)
Loss: 1.390 | Acc: 49.073% (2699/5500)
Loss: 1.387 | Acc: 49.083% (2945/6000)
Loss: 1.389 | Acc: 49.000% (3185/6500)
Loss: 1.393 | Acc: 49.000% (3430/7000)
Loss: 1.389 | Acc: 48.920% (3669/7500)
Loss: 1.388 | Acc: 49.025% (3922/8000)
Loss: 1.392 | Acc: 48.871% (4154/8500)
Loss: 1.392 | Acc: 48.922% (4403/9000)
Loss: 1.390 | Acc: 49.095% (4664/9500)
Loss: 1.391 | Acc: 48.970% (4897/10000)
Saving..
Epoch: 6, Learning Rate: 0.0009969804777275899

Epoch: 7
Loss: 1.402 | Acc: 49.219% (252/512)
Loss: 1.473 | Acc: 46.875% (480/1024)
Loss: 1.441 | Acc: 48.112% (739/1536)
Loss: 1.455 | Acc: 47.559% (974/2048)
Loss: 1.451 | Acc: 47.656% (1220/2560)
Loss: 1.442 | Acc: 48.372% (1486/3072)
Loss: 1.468 | Acc: 47.098% (1688/3584)
Loss: 1.474 | Acc: 46.851% (1919/4096)
Loss: 1.473 | Acc: 46.788% (2156/4608)
Loss: 1.467 | Acc: 46.816% (2397/5120)
Lo

...

Loss: 0.622 | Acc: 77.781% (23496/30208)
Loss: 0.623 | Acc: 77.718% (23875/30720)
Loss: 0.623 | Acc: 77.709% (24270/31232)
Loss: 0.623 | Acc: 77.722% (24672/31744)
Loss: 0.624 | Acc: 77.700% (25063/32256)
Loss: 0.624 | Acc: 77.731% (25471/32768)
Loss: 0.625 | Acc: 77.710% (25862/33280)
Loss: 0.623 | Acc: 77.743% (26271/33792)
Loss: 0.623 | Acc: 77.769% (26678/34304)
Loss: 0.624 | Acc: 77.720% (27059/34816)
Loss: 0.624 | Acc: 77.732% (27461/35328)
Loss: 0.623 | Acc: 77.762% (27870/35840)
Loss: 0.625 | Acc: 77.699% (28245/36352)
Loss: 0.625 | Acc: 77.686% (28638/36864)
Loss: 0.624 | Acc: 77.702% (29042/37376)
Loss: 0.625 | Acc: 77.666% (29426/37888)
Loss: 0.625 | Acc: 77.693% (29834/38400)
Loss: 0.625 | Acc: 77.668% (30222/38912)
Loss: 0.627 | Acc: 77.592% (30590/39424)
Loss: 0.626 | Acc: 77.627% (31001/39936)
Loss: 0.626 | Acc: 77.626% (31398/40448)
Loss: 0.626 | Acc: 77.610% (31789/40960)
Loss: 0.626 | Acc: 77.599% (32182/41472)
Loss: 0.625 | Acc: 77.622% (32589/41984)
Loss: 0.625 | Ac

Loss: 0.616 | Acc: 77.797% (9958/12800)
Loss: 0.612 | Acc: 77.930% (10374/13312)
Loss: 0.614 | Acc: 77.879% (10766/13824)
Loss: 0.617 | Acc: 77.790% (11152/14336)
Loss: 0.616 | Acc: 77.889% (11565/14848)
Loss: 0.615 | Acc: 77.956% (11974/15360)
Loss: 0.616 | Acc: 77.898% (12364/15872)
Loss: 0.616 | Acc: 77.917% (12766/16384)
Loss: 0.620 | Acc: 77.853% (13154/16896)
Loss: 0.619 | Acc: 77.884% (13558/17408)
Loss: 0.619 | Acc: 77.930% (13965/17920)
Loss: 0.617 | Acc: 78.000% (14377/18432)
Loss: 0.617 | Acc: 77.993% (14775/18944)
Loss: 0.617 | Acc: 78.017% (15179/19456)
Loss: 0.617 | Acc: 78.020% (15579/19968)
Loss: 0.616 | Acc: 78.052% (15985/20480)
Loss: 0.617 | Acc: 78.030% (16380/20992)
Loss: 0.617 | Acc: 78.065% (16787/21504)
Loss: 0.617 | Acc: 78.061% (17186/22016)
Loss: 0.616 | Acc: 78.085% (17591/22528)
Loss: 0.617 | Acc: 78.051% (17983/23040)
Loss: 0.618 | Acc: 77.993% (18369/23552)
Loss: 0.619 | Acc: 77.955% (18759/24064)
Loss: 0.619 | Acc: 77.938% (19154/24576)
Loss: 0.619 | Acc

Loss: 0.817 | Acc: 72.745% (4001/5500)
Loss: 0.816 | Acc: 72.900% (4374/6000)
Loss: 0.815 | Acc: 72.923% (4740/6500)
Loss: 0.822 | Acc: 72.643% (5085/7000)
Loss: 0.818 | Acc: 72.667% (5450/7500)
Loss: 0.820 | Acc: 72.737% (5819/8000)
Loss: 0.821 | Acc: 72.694% (6179/8500)
Loss: 0.822 | Acc: 72.700% (6543/9000)
Loss: 0.819 | Acc: 72.705% (6907/9500)
Loss: 0.819 | Acc: 72.710% (7271/10000)
Epoch: 88, Learning Rate: 0.0005859645501397043

Epoch: 89
Loss: 0.641 | Acc: 74.609% (382/512)
Loss: 0.608 | Acc: 76.758% (786/1024)
Loss: 0.609 | Acc: 77.279% (1187/1536)
Loss: 0.606 | Acc: 77.441% (1586/2048)
Loss: 0.597 | Acc: 78.281% (2004/2560)
Loss: 0.593 | Acc: 78.516% (2412/3072)
Loss: 0.599 | Acc: 78.237% (2804/3584)
Loss: 0.592 | Acc: 78.369% (3210/4096)
Loss: 0.594 | Acc: 78.190% (3603/4608)
Loss: 0.600 | Acc: 77.988% (3993/5120)
Loss: 0.600 | Acc: 78.125% (4400/5632)
Loss: 0.607 | Acc: 77.962% (4790/6144)
Loss: 0.604 | Acc: 78.005% (5192/6656)
Loss: 0.608 | Acc: 77.860% (5581/7168)
Loss: 0

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x0000029B6739BA30>
Traceback (most recent call last):
  File "C:\Users\mrtz\anaconda3\envs\mtp\lib\site-packages\torch\utils\data\dataloader.py", line 1510, in __del__
    self._shutdown_workers()
  File "C:\Users\mrtz\anaconda3\envs\mtp\lib\site-packages\torch\utils\data\dataloader.py", line 1468, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


KeyboardInterrupt: 